In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_TRACING"] = 'true'
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model

google_llm = init_chat_model("google_genai:gemini-2.5-flash-lite")
google_llm

s:\langchain_new_version\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x0000023B33F30AD0>, default_metadata=(), model_kwargs={})

In [7]:
from typing_extensions import TypedDict, Annotated

class MoviesDict(TypedDict):
    """A Movie with details"""
    title: Annotated[str, ..., "The movie title"]
    year: Annotated[int, ..., "The year the movie was released"]
    rating: Annotated[float, ..., "The movie rating"]
    genre: Annotated[str, ..., "The movie genre"]   

model_with_structure = google_llm.with_structured_output(
    MoviesDict
)
model_with_structure


RunnableBinding(bound=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x0000023B33F30AD0>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'title': 'MoviesDict', 'description': 'A Movie with details', 'type': 'object', 'properties': {'title': {'description': 'The movie title', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'rating': {'description': 'The movie rating', 'ty

In [9]:
result = model_with_structure.invoke("Tell me about avengers movie")
result

{'title': 'Avengers', 'year': 2012, 'rating': 8.0, 'genre': 'Superhero'}

In [10]:
### also will use nested struture

In [11]:
google_llm.profile

{'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}

In [14]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

groq_llm = init_chat_model("groq:openai/gpt-oss-120b")
groq_llm



@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model=groq_llm,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')